## REQUIRES ENV FILE WITH AGOL CREDENTIALS

In [1]:
from typing import List, Sequence, Tuple, Literal
from math import hypot

from shapely.geometry import (
    Point as ShapelyPoint,
    LineString as ShapelyLineString,
    MultiLineString as ShapelyMultiLineString,
    Polygon as ShapelyPolygon,
    MultiPolygon as ShapelyMultiPolygon,
)
from shapely.ops import transform
import pyproj


from typing import List, Sequence, Tuple, Literal
from shapely.geometry import (
    Point as ShapelyPoint,
    LineString as ShapelyLineString,
    MultiLineString as ShapelyMultiLineString,
    Polygon as ShapelyPolygon,
    MultiPolygon as ShapelyMultiPolygon,
)
from shapely.ops import transform, unary_union
import pyproj


from shapely.geometry import (
    Point as ShapelyPoint,
    LineString as ShapelyLineString,
    Polygon as ShapelyPolygon,
    MultiPolygon,
)
from shapely.ops import transform, unary_union, polygonize
import pyproj
import json
import requests
import math
import streamlit as st
import logging
from shapely.geometry import LineString
from shapely.ops import unary_union, linemerge
from typing import Tuple, Optional, List, Dict, Any

In [2]:
import os
import requests
from dotenv import load_dotenv

# Load .env file once at app startup (safe to call multiple times)
load_dotenv()


def get_agol_token() -> str:
    """
    Generates an authentication token for ArcGIS Online using a username and password
    stored in environment variables.

    Environment variables required:
        - AGOL_USERNAME
        - AGOL_PASSWORD

    Returns:
        str: A valid authentication token used to make authorized API requests.

    Raises:
        ValueError: If credentials are missing or authentication fails.
        ConnectionError: If requests cannot reach the AGOL endpoint.
    """
    username = os.getenv("AGOL_USERNAME")
    password = os.getenv("AGOL_PASSWORD")

    if not username or not password:
        raise ValueError("Missing AGOL_USERNAME or AGOL_PASSWORD in environment variables.")

    url = "https://www.arcgis.com/sharing/rest/generateToken"

    data = {
        "username": username,
        "password": password,
        "referer": "https://www.arcgis.com",
        "f": "json"
    }

    try:
        response = requests.post(url, data=data)

        if response.status_code != 200:
            raise Exception(
                f"Request failed with status code {response.status_code}: {response.text}"
            )

        token_data = response.json()

        if "token" in token_data:
            return token_data["token"]
        elif "error" in token_data:
            raise ValueError(
                f"Authentication failed: {token_data['error']['message']}"
            )
        else:
            raise ValueError("Unexpected response format: Token not found.")

    except requests.exceptions.RequestException as e:
        raise ConnectionError(f"Failed to connect to ArcGIS Online: {e}")


In [3]:
def query_record(url: str, layer: int, where: str, fields="*", return_geometry=False):
    """
    Executes an SQL-style query against an ArcGIS REST API layer and returns matching records.

    Parameters:
        url: str
            FeatureServer base URL (may or may not already include a layer segment).
        layer: int
            Layer index when url is a FeatureServer root.
        where: str
            SQL-like filter clause (e.g., "GlobalID='...'" or "1=1").
        fields: str
            outFields string. "*" requests all fields.
        return_geometry: bool
            Whether to return geometry in results.

    Returns:
        list: List of 'features' from the ArcGIS REST response.
    """
    token = get_agol_token()
    if not token:
        raise ValueError("Authentication failed: Invalid token.")


    # If the URL already ends with the layer number, don't add it again
    if url.split("/")[-1].isdigit():
        query_url = f"{url}/query"
    else:
        query_url = f"{url}/{layer}/query"

    params = {
        "where": where,
        "outFields": fields,
        "returnGeometry": str(return_geometry).lower(),
        "outSR": 4326,
        "f": "json",
        "token": token
    }

    response = requests.get(query_url, params=params)
    if response.status_code != 200:
        raise Exception(
            f"Request failed with status code {response.status_code}: {response.text}"
        )

    data = response.json()
    if "error" in data:
        raise Exception(
            f"API Error: {data['error']['message']} - {data['error'].get('details', [])}"
        )

    return data.get("features", [])

In [4]:
token = get_agol_token()

traffic_impact_url = "https://services.arcgis.com/r4A0V7UzH9fcLVvv/arcgis/rest/services/service_001ad60c04114d70a690d24123bb6742/FeatureServer"

traffic_impact_layers = {
'traffic_impacts_layer': 0,
'traffic_impact_routes_layer': 1,
'traffic_impact_start_points_layer': 2,
'traffic_impact_end_points_layer': 3
}


In [5]:
events = query_record(
    traffic_impact_url,
    traffic_impact_layers['traffic_impacts_layer'],
    '1=1',
    return_geometry=True
)



for event in events:
    objectid = event['attributes']['objectid']

    updates = []

    package = {
        "attributes": {
            "objectid": objectid,
            "Event_Name": "Blank Traffic Impact",
            "Route_ID": "NIE",
            "Route_Name": "NIE",
            "Start_X": 0.0,
            "Start_Y": 0.0,
            "End_X": 0.0,
            "End_Y": 0.0,
            "DOT_PF_Proj_Phone_COMM": "NIE",
            "Agency_Name_COMM": "NIE",
            "Agency_Phone_COMM": "NIE",
            "Contractor_Name_COMM": "NIE",
            "Contractor_Phone_COMM": "NIE",
            "Event_Type_COMM": "Roadwork / Maintenance",
            "Full_Closure_COMM": "NIE",
            "Status_COMM": "NIE",
            "Description_COMM": "NIE",
            "Broadcast_COMM": "NIE",
            "DOT_PF_Proj_Phone_PUBLIC": "",
            "Agency_Name_PUBLIC": "",
            "Agency_Phone_PUBLIC": "",
            "Contractor_Name_PUBLIC": "",
            "Contractor_Phone_PUBLIC": "",
            "Event_Type_PUBLIC": "",
            "Full_Closure_PUBLIC": "",
            "Status_PUBLIC": "",
            "Description_PUBLIC": "",
            "Broadcast_PUBLIC": "",
            "DOT_PF_Proj_Phone": "",
            "Agency_Name": "",
            "Agency_Phone": "",
            "Contractor_Name": "",
            "Contractor_Phone": "",
            "Event_Type": "",
            "Full_Closure": "",
            "Status": "",
            "Description": "",
            "Broadcast": "",
            "Notes_for_Approver": "No Notes",
            "Notes_for_Next_Week": "No Notes",
            "Drafter": "Unassigned",
            "Approver": "Unassigned",
            "Alaska_511_Comm": "NIE",
            "Log_Status": "Blank",
            "Agency_Name": "DOT&PF",
            "Alaska_511": '',
            "Drafter": "Unassigned",
            "Drafter_Email": "",
            "Approver": "Unassigned",
            "Approver_Email": '',
            "Last_Submitter": ''
        },
        "geometry": event.get("geometry")
    }

    updates.append(package)

    # --------------------------------------------------
    # APPLY EDITS (UPDATES)
    # --------------------------------------------------

    apply_edits_url = (
        f"{traffic_impact_url}/{traffic_impact_layers['traffic_impacts_layer']}/applyEdits"
    )

    payload = {
        "f": "json",
        "updates": json.dumps(updates),
        "token": token
    }

    response = requests.post(apply_edits_url, data=payload)

    print("Status:", response.status_code)
    print(json.dumps(response.json(), indent=2))


Status: 200
{
  "addResults": [],
  "updateResults": [
    {
      "objectId": 14,
      "uniqueId": 14,
      "globalId": null,
      "success": true
    }
  ],
  "deleteResults": []
}
Status: 200
{
  "addResults": [],
  "updateResults": [
    {
      "objectId": 15,
      "uniqueId": 15,
      "globalId": null,
      "success": true
    }
  ],
  "deleteResults": []
}
Status: 200
{
  "addResults": [],
  "updateResults": [
    {
      "objectId": 16,
      "uniqueId": 16,
      "globalId": null,
      "success": true
    }
  ],
  "deleteResults": []
}
Status: 200
{
  "addResults": [],
  "updateResults": [
    {
      "objectId": 17,
      "uniqueId": 17,
      "globalId": null,
      "success": true
    }
  ],
  "deleteResults": []
}
Status: 200
{
  "addResults": [],
  "updateResults": [
    {
      "objectId": 18,
      "uniqueId": 18,
      "globalId": null,
      "success": true
    }
  ],
  "deleteResults": []
}
Status: 200
{
  "addResults": [],
  "updateResults": [
    {
      "ob